<a href="https://colab.research.google.com/github/smuziVpyze/russian-market-news-analysis/blob/main/06c_ticker_filtered.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Устройство: {device}')

PROJECT_DIR = '/content/drive/MyDrive/russian_market_news_analysis'

TICKERS = ['MOEX', 'SBER', 'GAZP', 'LKOH', 'PLZL', 'YDEX']

# Загружаем новости с сентиментом
df_news = pd.read_csv(f'{PROJECT_DIR}/data/processed/news_sentiment.csv')
df_news['date'] = pd.to_datetime(df_news['date'])

# Загружаем рыночные признаки
datasets = {}
for ticker in TICKERS:
    df = pd.read_csv(f'{PROJECT_DIR}/data/features/{ticker}_features.csv')
    df['date'] = pd.to_datetime(df['date'])
    datasets[ticker] = df

print(f'✅ Новостей: {len(df_news):,}')
print(f'✅ Тикеров: {len(datasets)}')

Mounted at /content/drive
Устройство: cuda
✅ Новостей: 61,398
✅ Тикеров: 6


In [2]:
# Ключевые слова для каждого тикера
TICKER_KEYWORDS = {
    'MOEX': ['московская биржа', 'мосбиржа', 'moex', 'imoex'],
    'SBER': ['сбер', 'сбербанк', 'греф'],
    'GAZP': ['газпром', 'газпрома', 'газпроме'],
    'LKOH': ['лукойл', 'лукойла', 'лукойле'],
    'PLZL': ['полюс', 'polyus'],
    'YDEX': ['яндекс', 'yandex'],
}

df_news['text_lower'] = df_news['full_text'].str.lower()

# Считаем сентимент только по релевантным новостям
ticker_sentiment = {}

for ticker, keywords in TICKER_KEYWORDS.items():
    mask = df_news['text_lower'].str.contains('|'.join(keywords), na=False)
    df_ticker_news = df_news[mask].copy()

    # Агрегация по дням
    daily = df_ticker_news.groupby('date').agg(
        news_count_filtered=('sentiment_score', 'count'),
        sentiment_mean_filtered=('sentiment_score', 'mean'),
        sentiment_net_filtered=('sentiment_label', lambda x:
            ((x=='positive').sum() - (x=='negative').sum()) / len(x)),
        positive_ratio_filtered=('sentiment_label', lambda x:
            (x=='positive').sum() / len(x)),
        negative_ratio_filtered=('sentiment_label', lambda x:
            (x=='negative').sum() / len(x)),
    ).reset_index()

    # Лаги
    for lag in [1, 2, 3]:
        daily[f'sent_filtered_lag{lag}'] = daily['sentiment_mean_filtered'].shift(lag)
        daily[f'net_filtered_lag{lag}']  = daily['sentiment_net_filtered'].shift(lag)

    # Скользящие средние
    daily['sent_filtered_ma3'] = daily['sentiment_mean_filtered'].rolling(3).mean()
    daily['sent_filtered_ma7'] = daily['sentiment_mean_filtered'].rolling(7).mean()

    ticker_sentiment[ticker] = daily
    print(f'{ticker}: {mask.sum():,} релевантных новостей, '
          f'{len(daily)} дней с данными')

MOEX: 4,065 релевантных новостей, 586 дней с данными
SBER: 2,571 релевантных новостей, 624 дней с данными
GAZP: 3,367 релевантных новостей, 660 дней с данными
LKOH: 1,381 релевантных новостей, 548 дней с данными
PLZL: 448 релевантных новостей, 248 дней с данными
YDEX: 1,440 релевантных новостей, 497 дней с данными


In [3]:
MARKET_FEATURES = [
    'return_lag1', 'return_lag2', 'return_lag3',
    'return_2d', 'return_5d', 'volatility_5d', 'volatility_20d',
    'rsi', 'price_to_ma5', 'price_to_ma20', 'volume_ratio'
]

FILTERED_SENTIMENT_FEATURES = [
    'sentiment_mean_filtered', 'sentiment_net_filtered',
    'positive_ratio_filtered', 'negative_ratio_filtered',
    'news_count_filtered',
    'sent_filtered_lag1', 'net_filtered_lag1',
    'sent_filtered_lag2', 'net_filtered_lag2',
    'sent_filtered_lag3', 'net_filtered_lag3',
    'sent_filtered_ma3', 'sent_filtered_ma7',
]

ALL_FEATURES = MARKET_FEATURES + FILTERED_SENTIMENT_FEATURES

datasets_filtered = {}

for ticker in TICKERS:
    df_market = datasets[ticker]
    df_sent   = ticker_sentiment[ticker]

    df_merged = pd.merge(df_market, df_sent, on='date', how='inner')
    df_merged = df_merged.dropna(subset=ALL_FEATURES + ['target'])
    df_merged = df_merged.reset_index(drop=True)

    datasets_filtered[ticker] = df_merged
    print(f'{ticker}: {len(df_merged)} строк после объединения')

MOEX: 495 строк после объединения
SBER: 514 строк после объединения
GAZP: 528 строк после объединения
LKOH: 465 строк после объединения
PLZL: 222 строк после объединения
YDEX: 392 строк после объединения


In [4]:
class TimeSeriesDataset(Dataset):
    def __init__(self, X, y, seq_len=10):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y)
        self.seq_len = seq_len
    def __len__(self):
        return len(self.X) - self.seq_len
    def __getitem__(self, idx):
        return self.X[idx:idx+self.seq_len], self.y[idx+self.seq_len]

class LSTMClassifier(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.5):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                            batch_first=True, dropout=dropout)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 32), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(32, 1), nn.Sigmoid()
        )
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.classifier(out[:, -1, :]).squeeze()

def train_lstm_filtered(ticker, seq_len=10, epochs=100, patience=15):
    df = datasets_filtered[ticker]
    X = df[ALL_FEATURES].values
    y = df['target'].values

    split_idx = int(len(X) * 0.8)
    X_train, X_test = X[:split_idx], X[split_idx:]
    y_train, y_test = y[:split_idx], y[split_idx:]

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test  = scaler.transform(X_test)

    train_ds = TimeSeriesDataset(X_train, y_train, seq_len)
    test_ds  = TimeSeriesDataset(X_test,  y_test,  seq_len)
    train_dl = DataLoader(train_ds, batch_size=16, shuffle=False, drop_last=True)
    test_dl  = DataLoader(test_ds,  batch_size=16, shuffle=False, drop_last=False)

    model = LSTMClassifier(len(ALL_FEATURES)).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
    criterion = nn.BCELoss()

    best_auc, best_state, patience_counter = 0, None, 0

    for epoch in range(epochs):
        model.train()
        for X_batch, y_batch in train_dl:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            loss = criterion(model(X_batch), y_batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        model.eval()
        all_probs, all_true = [], []
        with torch.no_grad():
            for X_batch, y_batch in test_dl:
                probs = np.atleast_1d(model(X_batch.to(device)).cpu().numpy())
                all_probs.extend(probs.tolist())
                all_true.extend(y_batch.numpy().tolist())

        auc = roc_auc_score(all_true, all_probs)
        if auc > best_auc:
            best_auc = auc
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1

        if (epoch+1) % 10 == 0:
            print(f'  Epoch {epoch+1:3d} | auc={auc:.3f} | best={best_auc:.3f} | patience={patience_counter}')

        if patience_counter >= patience:
            print(f'  Early stopping на эпохе {epoch+1}')
            break

    model.load_state_dict(best_state)
    model.eval()
    all_probs, all_true = [], []
    with torch.no_grad():
        for X_batch, y_batch in test_dl:
            probs = np.atleast_1d(model(X_batch.to(device)).cpu().numpy())
            all_probs.extend(probs.tolist())
            all_true.extend(y_batch.numpy().tolist())

    all_preds = (np.array(all_probs) > 0.5).astype(int)
    return {
        'ticker':   ticker,
        'model':    'lstm_filtered',
        'accuracy': accuracy_score(all_true, all_preds),
        'f1':       f1_score(all_true, all_preds, zero_division=0),
        'roc_auc':  roc_auc_score(all_true, all_probs),
    }

# Запускаем
print('=== LSTM с фильтрацией новостей по тикеру ===\n')
lstm_filtered_results = []
for ticker in TICKERS:
    print(f'--- {ticker} ---')
    result = train_lstm_filtered(ticker)
    lstm_filtered_results.append(result)
    print(f'✅ {ticker}: accuracy={result["accuracy"]:.3f}, '
          f'f1={result["f1"]:.3f}, roc_auc={result["roc_auc"]:.3f}\n')

=== LSTM с фильтрацией новостей по тикеру ===

--- MOEX ---
  Epoch  10 | auc=0.462 | best=0.462 | patience=0
  Epoch  20 | auc=0.497 | best=0.497 | patience=0
  Epoch  30 | auc=0.339 | best=0.497 | patience=10
  Early stopping на эпохе 35
✅ MOEX: accuracy=0.517, f1=0.394, roc_auc=0.497

--- SBER ---
  Epoch  10 | auc=0.566 | best=0.589 | patience=5
  Epoch  20 | auc=0.538 | best=0.589 | patience=15
  Early stopping на эпохе 20
✅ SBER: accuracy=0.430, f1=0.602, roc_auc=0.589

--- GAZP ---
  Epoch  10 | auc=0.522 | best=0.533 | patience=2
  Epoch  20 | auc=0.538 | best=0.538 | patience=0
  Epoch  30 | auc=0.521 | best=0.555 | patience=1
  Epoch  40 | auc=0.449 | best=0.575 | patience=3
  Epoch  50 | auc=0.463 | best=0.575 | patience=13
  Early stopping на эпохе 52
✅ GAZP: accuracy=0.521, f1=0.582, roc_auc=0.575

--- LKOH ---
  Epoch  10 | auc=0.458 | best=0.468 | patience=9
  Epoch  20 | auc=0.508 | best=0.548 | patience=4
  Epoch  30 | auc=0.522 | best=0.548 | patience=14
  Early stopp

In [5]:
# Загружаем предыдущие результаты
df_all = pd.read_csv(f'{PROJECT_DIR}/models/all_model_results.csv')
df_lstm_best = df_all[df_all['model'] == 'lstm_best'][['ticker', 'model', 'roc_auc']]

df_filtered = pd.DataFrame(lstm_filtered_results)[['ticker', 'model', 'roc_auc']]

print('=== СРАВНЕНИЕ: общий сентимент vs фильтрованный ===\n')
print(f'{"Тикер":<6} {"LSTM (общий)":<15} {"LSTM (фильтр)":<15} {"Разница"}')
print('-' * 50)

for ticker in TICKERS:
    auc_general  = df_lstm_best[df_lstm_best['ticker']==ticker]['roc_auc'].values[0]
    auc_filtered = df_filtered[df_filtered['ticker']==ticker]['roc_auc'].values[0]
    diff = auc_filtered - auc_general
    arrow = '↑' if diff > 0 else '↓'
    print(f'{ticker:<6} {auc_general:<15.3f} {auc_filtered:<15.3f} {arrow}{abs(diff):.3f}')

=== СРАВНЕНИЕ: общий сентимент vs фильтрованный ===

Тикер  LSTM (общий)    LSTM (фильтр)   Разница
--------------------------------------------------
MOEX   0.591           0.497           ↓0.094
SBER   0.618           0.589           ↓0.029
GAZP   0.655           0.575           ↓0.079
LKOH   0.564           0.548           ↓0.016
PLZL   0.625           0.560           ↓0.065
YDEX   0.586           0.460           ↓0.127
